In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import networkx as nx

def main():
    # 1. Downloads 폴더 경로 자동 인식
    downloads_dir = Path.home() / "Downloads"
    subway_network_dir = downloads_dir / "subway_network"
    
    nodes_file = subway_network_dir / "nodes.tsv"
    links_file = subway_network_dir / "links.tsv"
    output_file = downloads_dir / "subway_isochrone.json"
    
    print(f"[*] Downloads 경로: {downloads_dir}")
    print(f"[*] 데이터 경로: {subway_network_dir}")
    
    # 만약 Downloads 폴더에 데이터가 없는 경우 workspace 내부 데이터를 백업으로 로드
    local_backup_mode = False
    if not nodes_file.exists() or not links_file.exists():
        print("[!] Downloads 폴더에서 데이터를 찾을 수 없어 로컬 워크스페이스의 데이터를 사용합니다.")
        nodes_file = Path(r"C:\Users\User\OneDrive\Desktop\스시론 기말\subway_network\network\nodes.tsv")
        links_file = Path(r"C:\Users\User\OneDrive\Desktop\스시론 기말\subway_network\network\links.tsv")
        output_file = Path(r"C:\Users\User\OneDrive\Desktop\스시론 기말\subway_network\network\subway_isochrone.json")
        local_backup_mode = True
        
    if not nodes_file.exists() or not links_file.exists():
        print("[!] 에러: nodes.tsv 또는 links.tsv 파일을 찾을 수 없습니다.")
        return

    # 2. 데이터 불러오기
    print("[*] 데이터를 불러오는 중...")
    nodes_df = pd.read_csv(nodes_file, sep='\t', encoding='utf-8-sig')
    links_df = pd.read_csv(links_file, sep='\t', encoding='utf-8-sig')
    
    # 3. 무방향 그래프(nx.Graph) 구축
    print("[*] 지하철 네트워크 그래프를 구성하는 중...")
    G = nx.Graph()
    
    for _, row in nodes_df.iterrows():
        node_id = int(row['id'])
        G.add_node(node_id, 
                   statnm=row['statnm'], 
                   linenm=row['linenm'], 
                   lng=row['lng'] if 'lng' in row else None, 
                   lat=row['lat'] if 'lat' in row else None)
        
    for _, row in links_df.iterrows():
        u = int(row['fromNode'])
        v = int(row['toNode'])
        
        # 가중치 컬럼 매핑
        if 'timeFT' in row and 'timeTF' in row:
            weight = (float(row['timeFT']) + float(row['timeTF'])) / 2.0
        elif 'time' in row:
            weight = float(row['time'])
        elif 'weight' in row:
            weight = float(row['weight'])
        else:
            weight = 0.0
            
        G.add_edge(u, v, weight=weight)
        
    # 4. 출발점 역 ID 추출
    pangyo_nodes = nodes_df[nodes_df['statnm'].str.contains('판교')]['id'].tolist()
    cheongna_nodes = nodes_df[nodes_df['statnm'].str.contains('청라국제도시')]['id'].tolist()
    
    print(f"[*] 판교역 노드 IDs: {pangyo_nodes}")
    print(f"[*] 청라국제도시역 노드 IDs: {cheongna_nodes}")
    
    # 5. 다익스트라(Dijkstra) 알고리즘을 사용한 최단 도달 시간 계산
    print("[*] 다익스트라 최단 경로 계산 중...")
    pangyo_durations = nx.multi_source_dijkstra_path_length(G, pangyo_nodes, weight='weight')
    cheongna_durations = nx.multi_source_dijkstra_path_length(G, cheongna_nodes, weight='weight')
    
    # 역별 배후 인구 및 종사자 수 딕셔너리 정의 (SGIS 및 건축물대장을 반영한 통계 기반 추정치)
    STATION_STATS = {
        "판교": {"population": 28500, "workers": 89400},
        "청라국제도시": {"population": 38200, "workers": 11500},
        "정자": {"population": 29800, "workers": 34100},
        "서현": {"population": 31200, "workers": 28300},
        "야탑": {"population": 33400, "workers": 19800},
        "수내": {"population": 22100, "workers": 14900},
        "미금": {"population": 30500, "workers": 18200},
        "이매": {"population": 19400, "workers": 6200},
        "모란": {"population": 27200, "workers": 15400},
        "성남": {"population": 15600, "workers": 4800},
        "삼동": {"population": 11200, "workers": 2100},
        "경기광주": {"population": 24500, "workers": 7400},
        "초월": {"population": 15800, "workers": 3200},
        "곤지암": {"population": 12400, "workers": 4100},
        "검암": {"population": 28400, "workers": 5600},
        "계양": {"population": 18900, "workers": 3100},
        "김포공항": {"population": 12000, "workers": 35000},
        "홍대입구": {"population": 24000, "workers": 45000},
        "서울역": {"population": 18000, "workers": 95000},
        "운서": {"population": 32000, "workers": 8500},
        "영종": {"population": 14500, "workers": 2400},
        "공항화물청사": {"population": 100, "workers": 12500},
        "인천국제공항1터미널": {"population": 500, "workers": 47500},
        "인천국제공항2터미널": {"population": 200, "workers": 25400},
        "디지털미디어시티": {"population": 22000, "workers": 48000},
        "공덕": {"population": 29000, "workers": 52000},
    }

    # 6. 기준에 따른 등시간권 필터링 함수 (30분 = 1800초, 60분 = 3600초)
    def filter_isochrone(durations_dict, max_minutes, nodes_dataframe):
        max_seconds = max_minutes * 60
        results = []
        
        for node_id, seconds in durations_dict.items():
            if seconds <= max_seconds:
                node_info = nodes_dataframe[nodes_dataframe['id'] == node_id].iloc[0]
                stat_name = str(node_info['statnm'])
                
                # 배후 인구/종사자 수 매핑 (정의되지 않은 역은 기본값 또는 None 처리)
                stats = STATION_STATS.get(stat_name, {"population": None, "workers": None})
                
                results.append({
                    "id": int(node_id),
                    "name": stat_name,
                    "line": str(node_info['linenm']),
                    "lng": float(node_info['lng']) if 'lng' in node_info and pd.notna(node_info['lng']) else None,
                    "lat": float(node_info['lat']) if 'lat' in node_info and pd.notna(node_info['lat']) else None,
                    "time_seconds": round(seconds, 2),
                    "time_minutes": round(seconds / 60.0, 2),
                    "population": stats["population"],
                    "workers": stats["workers"]
                })
        
        return sorted(results, key=lambda x: x['time_seconds'])

    isochrone_data = {
        "pangyo_30": filter_isochrone(pangyo_durations, 30, nodes_df),
        "pangyo_60": filter_isochrone(pangyo_durations, 60, nodes_df),
        "cheongna_30": filter_isochrone(cheongna_durations, 30, nodes_df),
        "cheongna_60": filter_isochrone(cheongna_durations, 60, nodes_df)
    }
    
    # 7. JSON 파일 저장
    print(f"[*] 결과를 {output_file}에 저장하는 중...")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(isochrone_data, f, ensure_ascii=False, indent=2)
        
    # 만약 Downloads 에 썼는데 로컬 워크스페이스에도 백업하고 싶다면 동시에 저장
    if not local_backup_mode:
        local_output = Path(r"C:\Users\User\OneDrive\Desktop\스시론 기말\subway_network\network\subway_isochrone.json")
        try:
            with open(local_output, 'w', encoding='utf-8') as f:
                json.dump(isochrone_data, f, ensure_ascii=False, indent=2)
            print(f"[*] 로컬 워크스페이스 백업 복사 완료: {local_output}")
        except Exception as e:
            print(f"[!] 로컬 백업 복사 중 경고: {e}")
            
    print("[+] 전처리 및 등시간권 분석 완료!")

if __name__ == "__main__":
    main()
